# More Than a Hack 2027 — 01 · Preprocessament (integrat amb el model)

Aquest notebook prepara les dades originals **només localment** i genera dos artefactes privats:

1. `data/processed/features_tactical.csv`: una fila per frame amb les posicions transformades, les distàncies, **les velocitats mitjanes dels dos equips** i les etiquetes tàctiques per frame.
2. `data/processed/smart_tagging_events.json`: intervals de `SET PIECES`, que el detector de fora de joc necessita per identificar els possibles reinicis sense haver de tornar a obrir l'XML original.

El notebook **no entrena cap model, no consulta el ground truth, no calcula la possessió i no accedeix a Internet**. No incloguis aquests dos fitxers en commits públics. Esborra les sortides de les cel·les abans de compartir el notebook.


## 1. Configuració i importacions

Desa aquest notebook dins de `solution/` al repositori de la hackató. No cal modificar els scripts oficials.

Si les dades es troben a la carpeta `data/` indicada per `games.py`, deixa `DATA_DIR_OVERRIDE = None`. Si les tens fora del repositori, indica la **carpeta arrel que conté `tactical/`**, per exemple `r"D:\HackathonDades"`. Aquest directori ha de quedar fora de l'abast dels agents d'IA.

In [ ]:
from pathlib import Path
import json
import sys
import re
import unicodedata
import numpy as np
import pandas as pd

# Si tens les dades en una ubicació externa, posa aquí la carpeta que conté tactical/.
DATA_DIR_OVERRIDE = None  # Exemple: r"D:\HackathonDades"
WRITE_CSV = True

# Detectem l'arrel del repositori independentment de si executeu des de l'arrel o solution/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "games.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No trobo games.py ni starter/. Obre el notebook des de l'arrel del repositori o solution/."
    )
sys.path.insert(0, str(ROOT))

from games import GAME, GAMES, DATA_DIR, source_frames
from starter import baseline, load_atd, load_smart_tagging

FPS = GAMES[GAME]["fps"]
PITCH_LENGTH_M, PITCH_WIDTH_M = GAMES[GAME]["pitch"]
N_FRAMES = source_frames("tactical", GAME)
DATA_ROOT = Path(DATA_DIR_OVERRIDE or DATA_DIR).expanduser().resolve()
OUTPUT_PATH = DATA_ROOT / "processed" / "features_tactical.csv"
EVENTS_PATH = DATA_ROOT / "processed" / "smart_tagging_events.json"

print(f"Joc: {GAME} | FPS: {FPS} | Frames esperats: {N_FRAMES:,}")
print("El processament es fa en local; no es farà cap petició de xarxa.")


## 2. Llegir el tracking **una sola vegada**

Cada registre ATD conté les posicions i velocitats dels jugadors i de la pilota. L'XML indica a quin equip correspon cada *track slot*.

Ara també calculem `home_mean_speed_mps` i `away_mean_speed_mps` **durant aquesta mateixa lectura**, sense haver de passar de nou per tot el TXT amb `team_speed.py`.

Per mantenir la mateixa definició de l'algoritme original, fem una mitjana de les velocitats disponibles de cada equip, acotades a l'interval **0–12 m/s** per evitar velocitats aberrants. La velocitat mitjana pot calcular-se encara que la pilota no sigui visible.


In [ ]:
def parse_xyz(token):
    """Converteix 'x,y,s' (o 'NaN') a tres floats; s és opcional."""
    parts = token.strip().split(",")
    if len(parts) < 2:
        return np.nan, np.nan, np.nan
    try:
        x = float(parts[0])
        y = float(parts[1])
        speed = float(parts[2]) if len(parts) >= 3 else np.nan
        return x, y, speed
    except ValueError:
        raise ValueError("Registre ATD inesperat: coordenades no numèriques")


def parse_tracking_lines(lines, team_by_slot, n_frames, pitch, fps, max_player_speed=12.0):
    """Genera totes les features de tracking i velocitat mitjana en una sola lectura del TXT."""
    length, width = pitch
    rows = []
    expected_frame = 1
    team_by_slot = np.asarray(team_by_slot)
    if not np.isin(team_by_slot, [0, 1]).all():
        raise ValueError("team_by_slot ha de contenir 0=H o 1=A")

    for line in lines:
        if not line.strip():
            continue
        parts = line.rstrip("\r\n").split(":")
        if len(parts) != 3:
            raise ValueError(f"Format ATD incorrecte a la línia {expected_frame}")
        frame_id = int(parts[0])
        if frame_id != expected_frame:
            raise ValueError(f"Frames desalineats: esperava {expected_frame} i he rebut {frame_id}")
        frame = frame_id - 1  # ATD comença per 1 i la submissió per 0.
        expected_frame += 1

        players = parts[1].split(";")
        if len(players) != len(team_by_slot):
            raise ValueError(
                f"Frame {frame}: {len(players)} slots al TXT, "
                f"{len(team_by_slot)} a les metadades XML"
            )

        bx, by, bs = parse_xyz(parts[2])
        visible = np.isfinite(bx) and np.isfinite(by)
        bx_m = bx * length if visible else np.nan
        by_m = by * width if visible else np.nan
        home_min = away_min = np.inf
        home_speed = away_speed = np.nan
        n_home = n_away = n_home_2m = n_away_2m = 0
        speed_sum = [0.0, 0.0]     # Suma de velocitats de H i A en aquest frame.
        speed_count = [0, 0]       # Nombre de tracks amb velocitat disponible.

        for slot, raw_player in enumerate(players):
            px, py, ps = parse_xyz(raw_player)
            team = int(team_by_slot[slot])  # 0 = local; 1 = visitant.

            # Calculem la velocitat mitjana de cada equip (independentment de la pilota).
            if np.isfinite(ps):
                speed_sum[team] += float(np.clip(ps, 0.0, max_player_speed))
                speed_count[team] += 1

            if not (np.isfinite(px) and np.isfinite(py)):
                continue
            if team == 0:
                n_home += 1
            else:
                n_away += 1
            if not visible:
                continue

            # Distància entre la pilota i aquest jugador, convertida a metres.
            distance = np.hypot((px - bx) * length, (py - by) * width)
            if team == 0:
                n_home_2m += distance <= 2.0
                if distance < home_min:
                    home_min, home_speed = distance, ps
            else:
                n_away_2m += distance <= 2.0
                if distance < away_min:
                    away_min, away_speed = distance, ps

        home_mean = speed_sum[0] / speed_count[0] if speed_count[0] else np.nan
        away_mean = speed_sum[1] / speed_count[1] if speed_count[1] else np.nan
        rows.append((frame, bx, by, bs, visible, bx_m, by_m, n_home, n_away,
                     home_min if np.isfinite(home_min) else np.nan,
                     away_min if np.isfinite(away_min) else np.nan,
                     home_speed, away_speed, n_home_2m, n_away_2m,
                     home_mean, away_mean))

    if len(rows) != n_frames:
        raise ValueError(f"Fitxer incomplet: he llegit {len(rows)} frames i n'esperava {n_frames}")

    cols = ["frame", "ball_x_norm", "ball_y_norm", "ball_speed_mps_atd",
            "ball_visible", "ball_x_m", "ball_y_m", "n_home_tracked", "n_away_tracked",
            "home_nearest_m", "away_nearest_m", "home_nearest_speed_mps",
            "away_nearest_speed_mps", "n_home_within_2m", "n_away_within_2m",
            "home_mean_speed_mps", "away_mean_speed_mps"]
    df = pd.DataFrame.from_records(rows, columns=cols)
    df["frame"] = df["frame"].astype("int32")
    df["ball_visible"] = df["ball_visible"].astype(bool)
    df["time_s"] = df["frame"] / fps

    # Diferència de distàncies: positiu = local més proper; negatiu = visitant.
    df["distance_margin_away_minus_home_m"] = df["away_nearest_m"] - df["home_nearest_m"]
    home = df["home_nearest_m"].to_numpy(dtype=float)
    away = df["away_nearest_m"].to_numpy(dtype=float)
    nearest = np.full(len(df), "", dtype="<U1")
    nearest[np.isfinite(home) & ((home <= away) | ~np.isfinite(away))] = "H"
    nearest[np.isfinite(away) & ((away < home) | ~np.isfinite(home))] = "A"
    df["nearest_team_observed"] = nearest

    # Desplaçament només quan la pilota és visible en dos frames consecutius.
    bx_arr = df["ball_x_m"].to_numpy(dtype=float)
    by_arr = df["ball_y_m"].to_numpy(dtype=float)
    displacement = np.full(len(df), np.nan, dtype=float)
    good_pairs = np.isfinite(bx_arr[1:]) & np.isfinite(bx_arr[:-1]) & \
                 np.isfinite(by_arr[1:]) & np.isfinite(by_arr[:-1])
    displacement[1:][good_pairs] = np.hypot(
        bx_arr[1:][good_pairs] - bx_arr[:-1][good_pairs],
        by_arr[1:][good_pairs] - by_arr[:-1][good_pairs]
    )
    df["ball_displacement_m_between_frames"] = displacement
    df["ball_apparent_speed_mps"] = displacement * fps

    # Quants frames seguits fa que no es detecta la pilota.
    is_missing = ~df["ball_visible"]
    df["ball_missing_streak_frames"] = (
        is_missing.astype(int).groupby((~is_missing).cumsum()).cumsum().astype("int32")
    )
    return df


## 3. Prova amb dades **sintètiques**

Comprovem la conversió de frames, les distàncies, els `NaN` i l'equip més proper **sense tocar dades restringides**.

In [ ]:
from io import StringIO

# Només dades sintètiques (no provenen del partit).
sample = StringIO("""1:0.20,0.50,2;0.80,0.50,3:0.21,0.50,5
2:0.20,0.50,2;0.80,0.50,3:NaN
3:0.20,0.50,2;0.80,0.50,3:0.78,0.50,6
""")
test_df = parse_tracking_lines(sample, team_by_slot=[0, 1],
                               n_frames=3, pitch=(100.0, 50.0), fps=25)
assert test_df["frame"].tolist() == [0, 1, 2]
assert test_df["ball_visible"].tolist() == [True, False, True]
assert test_df["nearest_team_observed"].tolist() == ["H", "", "A"]
assert np.isclose(test_df.loc[0, "home_nearest_m"], 1.0)
assert test_df["ball_missing_streak_frames"].tolist() == [0, 1, 0]
assert np.isnan(test_df.loc[2, "ball_displacement_m_between_frames"])
assert np.allclose(test_df["home_mean_speed_mps"], [2, 2, 2])
assert np.allclose(test_df["away_mean_speed_mps"], [3, 3, 3])
print("Proves sintètiques del tracking i de velocitats mitjanes superades.")


## 4. Llegir el tracking real (en local)

Es fa servir `starter/load_atd.py` per localitzar els fitxers i `starter/baseline.py` per obtenir l'equip de cada *track slot*. No cal `dataset.to_df()` perquè així evitem carregar els ~9 milions de registres de jugador a memòria alhora.

Si surt un error de fitxer no trobat, revisa `DATA_DIR_OVERRIDE` o l'estructura `data/tactical/`.

In [ ]:
meta_path, raw_path = load_atd.atd_paths(game=GAME, data_dir=str(DATA_ROOT), source="tactical")
team_by_slot = baseline.team_of_each_track(meta_path)
print(f"S'han detectat {len(team_by_slot)} track slots en les metadades.")

with open(raw_path, encoding="utf-8-sig") as handle:
    features_tracking = parse_tracking_lines(
        handle, team_by_slot=team_by_slot, n_frames=N_FRAMES,
        pitch=(PITCH_LENGTH_M, PITCH_WIDTH_M), fps=FPS
    )
assert len(features_tracking) == N_FRAMES
assert features_tracking["frame"].iloc[0] == 0
assert features_tracking["frame"].iloc[-1] == N_FRAMES - 1
print("Tracking processat; frames consecutius i sense desalineació.")

## 5. Afegir Smart Tagging (intervals que poden coincidir)

Convertim els intervals tàctics en **columnes 0/1 per frame** (`tag_build_up_H`, `tag_progression_A`, etc.). Aquestes columnes poden ajudar el model a decidir entre H i A quan falta la pilota, però **només són indicis**, mai una resposta segura.

A més, desem **només els intervals de `SET PIECES`** al fitxer `smart_tagging_events.json`: en un CSV de 0/1 es pot perdre el límit exacte entre intervals de reinicis contigus o solapats. `deadball.py` necessita aquests límits per detectar l'instant del servei. No es torna a llegir l'XML al model.


In [ ]:
def slug(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

# Valor Team del Smart Tagging segons DATA.md. Cal revisar si el proveïdor el canvia.
TAG_TEAM_MAP = {
    "barcelona": "H",
    "fc barcelona": "H",
    "atletic club": "A",  # Valor publicat per Metrica
    "athletic club": "A",
    "n/a": "unknown",
    "": "unknown"
}


def attach_smart_tagging(df, events):
    n = len(df)
    codes = sorted({slug(e.get("code") or "unknown") for e in events})
    kinds = ("H", "A", "unknown")
    columns = {
        f"tag_{code}_{kind}": np.zeros(n, dtype=np.uint8)
        for code in codes for kind in kinds
    }
    # Signals auxiliars sense suposar que el joc està aturat a tot el segment.
    restart_types = sorted({slug(e.get("tags", {}).get("Type"))
                            for e in events if slug(e.get("code")) == "set_pieces"
                            and e.get("tags", {}).get("Type")})
    for kind in restart_types:
        columns[f"restart_type_{kind}"] = np.zeros(n, dtype=np.uint8)
    event_count = np.zeros(n, dtype=np.int16)
    invalid = []

    for ev in events:
        begin = max(0, int(ev["start_frame"]))
        end = min(n, int(ev["end_frame"]))  # exclòs
        if begin >= end:
            continue
        code = slug(ev.get("code") or "unknown")
        raw_team = (ev.get("tags", {}).get("Team") or "").strip().casefold()
        if raw_team not in TAG_TEAM_MAP:
            invalid.append(raw_team)
            continue
        team = TAG_TEAM_MAP[raw_team]
        columns[f"tag_{code}_{team}"][begin:end] = 1
        event_count[begin:end] += 1

        if code == "set_pieces":
            kind = slug(ev.get("tags", {}).get("Type") or "")
            if kind and f"restart_type_{kind}" in columns:
                columns[f"restart_type_{kind}"][begin:end] = 1

    if invalid:
        raise ValueError(f"Team de Smart Tagging desconegut: {sorted(set(invalid))}. "
                         "Revisa TAG_TEAM_MAP abans de continuar.")
    columns["tag_event_count"] = event_count
    return pd.concat([df, pd.DataFrame(columns, index=df.index)], axis=1)


# Test sintètic: dues etiquetes solapades, una ofensiva i una defensiva.
test_events = [
    {"code": "BUILD UP", "start_frame": 0, "end_frame": 2, "tags": {"Team": "Barcelona"}},
    {"code": "DEFENSIVE TRANSITION", "start_frame": 1, "end_frame": 3,
     "tags": {"Team": "Atletic Club"}}
]
smart_test = attach_smart_tagging(test_df, test_events)
assert smart_test.loc[1, "tag_build_up_H"] == 1
assert smart_test.loc[1, "tag_defensive_transition_A"] == 1
assert smart_test.loc[1, "tag_event_count"] == 2
print("Proves sintètiques de Smart Tagging superades.")

In [ ]:
# Llegim l'XML d'Smart Tagging una única vegada durant el preprocessament.
events = load_smart_tagging.load(game=GAME, data_dir=str(DATA_ROOT), fps=FPS)
features = attach_smart_tagging(features_tracking, events)
assert features["frame"].is_unique
assert np.array_equal(features["frame"].to_numpy(), np.arange(N_FRAMES))

# Guardem els intervals de serveis/aturades sense alterar-los ni inferir la possessió.
# deadball.py només necessita code, start_frame, end_frame i tags.Type.
set_piece_events = [
    {
        "code": ev["code"],
        "start_frame": int(ev["start_frame"]),
        "end_frame": int(ev["end_frame"]),
        "tags": {"Type": ev.get("tags", {}).get("Type")}
    }
    for ev in events if (ev.get("code") or "").strip().upper() == "SET PIECES"
]
print(f"Smart Tagging processat: {len(events)} intervals; "
      f"{len(set_piece_events)} reinicis; {len(features.columns)} columnes.")


## 6. Control de qualitat (sense ground truth)

Validem el nombre i la numeració de frames, les velocitats mitjanes i els indicadors tàctics. No mostrem files reals perquè no quedin desades a les sortides del notebook.


In [ ]:
assert len(features) == N_FRAMES
assert np.array_equal(features["frame"].to_numpy(), np.arange(N_FRAMES))
assert features["ball_visible"].dtype == bool
assert (features["ball_missing_streak_frames"] >= 0).all()
assert features["nearest_team_observed"].isin(["", "H", "A"]).all()
assert (features.loc[~features["ball_visible"], "nearest_team_observed"] == "").all()
assert (features["tag_event_count"] >= 0).all()
for side in ("home", "away"):
    speed = features[f"{side}_mean_speed_mps"]
    assert ((speed >= 0) & (speed <= 12) | speed.isna()).all()
assert all(ev["start_frame"] < ev["end_frame"] for ev in set_piece_events)

print("Controls estructurals completats correctament.")
print("Files:", len(features), "| Columnes:", len(features.columns))
print("Velocitats mitjanes disponibles per H i A.")


## 7. Exportar els dos fitxers de preprocessament (privats)

- `features_tactical.csv`: totes les característiques per frame, incloses velocitats mitjanes i indicadors tàctics.
- `smart_tagging_events.json`: intervals originals de reinici que `deadball.py` farà servir per localitzar aturades.

**Cap dels dos no ha d'anar al repositori públic.** Han de quedar sota `data/processed/` o en una carpeta local externa protegida. Si utilitzes dades fora del repositori, indica també la ruta a `run.py`.


In [ ]:
if WRITE_CSV:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    features.to_csv(OUTPUT_PATH, index=False, na_rep="")
    EVENTS_PATH.write_text(json.dumps(set_piece_events, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Preprocessament guardat en local:")
    print(" -", OUTPUT_PATH.name)
    print(" -", EVENTS_PATH.name)
else:
    print("WRITE_CSV=False: no s'ha desat cap fitxer; les dades queden en memòria.")


## 8. Execució del model

Des de l'arrel del repositori:

```powershell
python solution/run.py --name "The goats"
```

`run.py` només llegeix `features_tactical.csv` (incloses `home_mean_speed_mps` i `away_mean_speed_mps`) i `smart_tagging_events.json`. **No torna a llegir l'ATD ni l'XML d'Smart Tagging.** Si les dades són en una ubicació externa diferent de `games.DATA_DIR`, cal passar `--features` i `--events`.

Les columnes `tag_*` (fases tàctiques per equip) queden a la taula com a possible senyal, però el model actual no les fa servir.

Per publicar el notebook, primer usa *Clear All Outputs*. Ni les dades originals ni els fitxers processats ni els resultats amb dades restringides s'han de compartir amb agents d'IA al núvol.